# Protocol verification

Synthetic data. This notebook does not decide H1, H2, or H5.

The decision rule, the stratum, and the power gate are in `src/study/protocol.py`. The written note is https://chrislernunes.github.io/atlas/.


In [1]:
from pathlib import Path
import os
import sys
root = Path.cwd()
if root.name == "notebooks":
    os.chdir(root.parent)
    root = Path.cwd()
sys.path.insert(0, str(root / "src"))
import pandas as pd
from IPython.display import display
from study.backtest.walk_forward import walk_forward
from study.data.generate import make_synthetic_panel
from study.experiment.verify import decision_record
from study.protocol import STRATUM_CELLS, MDE_RELATIVE_FROBENIUS

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
print("stratum cells", len(STRATUM_CELLS))
print("relative MDE", MDE_RELATIVE_FROBENIUS)


stratum cells 16
relative MDE 0.15


## One live path

N=20, T=36, moderate equicorrelation, Gaussian, seed 0. Weights at each date use only earlier returns.


In [2]:
returns, meta = make_synthetic_panel(20, 96, "moderate", seed=0, tail="gaussian")
path = walk_forward(returns, meta["true_sigmas"], lookback=36, cost_bps=0)
summary = path.groupby("method")[["frobenius", "eig_l1", "risk_abs", "eff_bets", "max_weight", "turnover", "optimizer_failed"]].mean()
display(summary.round(4))
print("HRP linkage counts")
display(path.loc[path.method == "hrp", "linkage"].value_counts())


,frobenius,eig_l1,risk_abs,eff_bets,max_weight,turnover,optimizer_failed
method,,,,,,,
const_corr_mv,0.0040,0.0048,0.0000,4.7590,0.2500,0.0322,0.0
equal_weight,NaN,NaN,NaN,20.0000,0.0500,0.0083,0.0
factor_mv,0.0069,0.0077,0.0001,4.9103,0.2499,0.0409,0.0
hrp,0.0078,0.0110,0.0001,13.1712,0.1311,0.0790,0.0
inverse_vol,NaN,NaN,NaN,17.1978,0.0933,0.0175,0.0
lw_mv,0.0078,0.0110,0.0001,7.1574,0.2039,0.0558,0.0
sample_mv,0.0082,0.0128,0.0001,5.0450,0.2491,0.0597,0.0


HRP linkage counts


linkage
single     40
average    20
Name: count, dtype: int64

## Verification sample

Loaded from `outputs/verification`, which is produced by `python -m study.experiment.verify`. The status is recomputed here from that panel.


In [3]:
root = Path("outputs/verification")
panel = pd.read_csv(root / "verification_panel.csv")
pilot = pd.read_csv(root / "pilot_panel.csv")
decision = decision_record(panel, pilot)
display(pd.Series(decision).to_frame("value"))
base = panel[(panel["innovation"] == "gaussian") & (panel["shock"] == "") & (~panel["detector"].astype(bool))]
display(base.groupby("method")[["frobenius", "eff_bets", "sharpe_0", "sharpe_20"]].mean().round(4))


,value
h1_status,NOT_A_DECISION_SAMPLE
theta_h1,0.000562
theta_h1_p05,-0.000134
theta_h1_p95,0.001575
mean_sample_frobenius,0.008714
relative_h1,0.06449
pilot_sigma,0.000935
pilot_mean_sample_frobenius,0.008399
mde_abs,0.00126
power_at_design_alternative,1.0


,frobenius,eff_bets,sharpe_0,sharpe_20
method,,,,


## Tail and shock slices

These rows are not in Θ. A sign change here is a limit on the claim, not a new primary result.


In [4]:
slice_ = panel[(panel.n_assets == 20) & (panel.lookback == 36) & (panel.corr_type == "moderate") & (panel.method == "lw_mv")]
display(slice_.groupby(["innovation", "shock", "detector"])[["frobenius", "var"]].mean().round(6))


frobenius       var
innovation shock detector                     
gaussian   corr  False      0.012795  0.000453
                 True       0.012503  0.000458
           vol   False      0.031488  0.000857
                 True       0.030269  0.000906